# Libraries

In [1]:
import torch
import random
import numpy as np

from kaggle_secrets import UserSecretsClient
from huggingface_hub import snapshot_download
from huggingface_hub import login
from huggingface_hub import create_repo, upload_file

# Global Variables

In [ ]:
MODEL_ID = "EdgeCompress01/Llama-3.2-3B-Instruct-SparseGPT"
MODEL_DIR = "/kaggle/working/model"
REPO_ID = "EdgeCompress01/Llama-3.2-3B-Instruct-SparseGPT-GGUF"
model_name_in_repo = "Sparsity/70/Llama-3.2-3B-Instruct-SparseGPT-Q4_K_M.gguf"
SEED = 42

# Functions

In [3]:
# --- 2. REPRODUCIBILITY ---
def set_reproducibility(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # Ensure deterministic behavior in CuDNN
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print(f"Global seed set to {seed}")

# Set Seed

In [4]:
set_reproducibility(SEED)

Global seed set to 42


# Huggingface Login

In [5]:
# --- 3. HUGGING FACE AUTHENTICATION ---

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
login(token=hf_token)
print("Loged in")

Loged in


# Load Model

In [ ]:
# Download specifically from the Models/25 folder
snapshot_download(
    repo_id=MODEL_ID,
    local_dir=MODEL_DIR,
    allow_patterns=["Models/70/*"], # This isolates the download to your specific folder
)

Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

'/kaggle/working/model'

**Clone llama.cpp**

In [ ]:
!git clone https://github.com/ggerganov/llama.cpp.git
%cd llama.cpp

Cloning into 'llama.cpp'...
remote: Enumerating objects: 85542, done.
remote: Counting objects: 100% (194/194), done.
remote: Compressing objects: 100% (158/158), done.


**Build llama.cpp**

In [ ]:
!cmake -B build -DCMAKE_BUILD_TYPE=Release
!cmake --build build -j

# Quantization

**Convert from huggingface to gguf**

In [ ]:
!python convert_hf_to_gguf.py ../model/Models/70 \
    --outfile ../model/llama-f16.gguf \
    --outtype f16

**Free space**

In [ ]:
!rm -rf ../model/Models/70/*.safetensors
!rm -rf ../model/Models/70/tokenizer*
!rm -rf ../model/Models/70/config*

**Quantize**

In [ ]:
!./build/bin/llama-quantize \
    ../model/llama-f16.gguf \
    ../model/llama-q4.gguf \
    q4_k_m

**Free space**

In [ ]:
!rm ../model/llama-f16.gguf

# PUSH TO HUGGING FACE

In [ ]:
# Create repo
create_repo(REPO_ID, repo_type="model", exist_ok=True)

# Upload quantized model
upload_file(
    path_or_fileobj=f"{MODEL_DIR}/llama-q4.gguf",
    path_in_repo=model_name_in_repo,
    repo_id=REPO_ID,
    repo_type="model"
)

print("Upload complete to organization!")